# CSA0812 – Assignment 5
## A4 – Plan Debugging and Profiling

**Student:** Darshan A  
**Register Number:** 1973260011  
**Course Outcome:** CO5  
**Marks:** 4 | K4 – Analyse

**Purpose:** This notebook is organised as a separate submission for this rubric item. Written explanations are provided in text cells. Where implementation is required, the Python cells can be run in Google Colab.

## A4.1 – Debugging Plan (2 Marks)

The cleaning pipeline should be tested in small stages: reading the CSV, parsing dates and hours, converting numeric fields, applying validity limits, converting Fahrenheit to Celsius, resolving missing values and removing duplicates.

| Tool | How it will be used | Example |
|---|---|---|
| `print()` | Temporarily inspect a few rows and intermediate values | Print the first five raw rows and the rows rejected by a cleaning rule |
| `assert` | Check conditions that must always be true | `assert 0 <= hour <= 23`; `assert cleaned["station"].notna().all()` |
| `logging` | Record timestamped warnings/errors with file and row context | Log invalid dates, sensor values outside limits and file read failures |
| `pdb` / IDE debugger | Pause execution, step through a faulty row, inspect variables and call stack | Set a breakpoint in `clean_row()` when station is Guindy and pm25 is negative |

**Debugging workflow**

1. Reproduce the error with a small CSV containing one known faulty row.
2. Print the raw row and parsed dictionary to identify where the value changes.
3. Add assertions for assumptions such as valid hour range and temperature unit.
4. Replace temporary prints with `logging.warning()` or `logging.error()` so the issue and source row are recorded.
5. Use `pdb.set_trace()` or an IDE breakpoint to step through the conversion and imputation code.
6. Fix the defect, rerun the test and confirm that the expected row is accepted, repaired or rejected.

### Example debugging instrumentation

```python
import logging
import pdb

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(message)s"
)

def validate_hour(hour):
    assert isinstance(hour, int), "Hour must be an integer"
    if not 0 <= hour <= 23:
        logging.error("Invalid hour: %s", hour)
        return False
    return True

# During a focused debugging session, enable a breakpoint:
# pdb.set_trace()
```

Temporary `print()` statements are useful for a quick check. Assertions test assumptions during development. Logging is preferable for repeatable runs because messages can be filtered and saved. A debugger is useful when the sequence of operations matters.

## A4.2 – Profiling Plan (2 Marks)

Three tools are used for different measurement purposes.

| Tool | Purpose | Planned use |
|---|---|---|
| `time.perf_counter()` | Wall-clock duration of a complete operation | Measure full CSV cleaning, SQLite loading and chart generation |
| `timeit` | Repeat a small code operation many times | Compare conversion or normalisation functions with a stable median measurement |
| `cProfile` | Function-level profiling | Find which functions consume the most cumulative or total time in the full pipeline |

**Profiling steps**

1. Use the same input file, machine and cleaning rules for each test.
2. Measure the complete pipeline with `time.perf_counter()` before and after the operation.
3. Repeat small alternatives with `timeit` and compare the median or best stable result.
4. Run the complete program under `cProfile` and sort the results by cumulative time.
5. Identify the slowest function, inspect its algorithm and optimise only that part.
6. Run correctness assertions again after optimisation to ensure the result has not changed.

### Example timing and profiling pattern

```python
import time
import timeit
import cProfile
import pstats

start = time.perf_counter()
# clean_station_file("station.csv")
elapsed = time.perf_counter() - start
print("Pipeline seconds:", elapsed)

# Compare small operations (example only)
# print(timeit.timeit("sum(values)", setup="values=list(range(1000))", number=1000))

# Profile the complete pipeline
# cProfile.run("run_pipeline()", "profile_output.prof")
# pstats.Stats("profile_output.prof").sort_stats("cumulative").print_stats(15)
```

The comments indicate where the real pipeline function is called. Profiling should be measured on the same data and environment for meaningful comparison.

## Alternative Concept and Complexity Analysis

If `print()` were used for every debugging event instead of the logging module and a debugger, the program could generate thousands of console writes. Console output is relatively slow and can obscure the important messages. It also becomes difficult to disable every print consistently.

| Method | Runtime effect | How to switch off |
|---|---|---|
| Many `print()` calls | Output I/O can add noticeable time; exact overhead depends on console and environment | Remove or comment them, or guard them with a debug flag |
| Logging | Has formatting/handler overhead when enabled, but levels make it configurable | Set the logging level to `WARNING` or `ERROR`, or disable a handler |
| Debugger | Pauses execution and is not suitable for production runs | Remove breakpoints / do not invoke `pdb.set_trace()` |

The data-cleaning algorithm's asymptotic complexity does not change merely because logging is used. The practical wall-clock time can increase with the number and destination of messages. Logging provides selective control and durable records; a debugger provides interactive inspection. Temporary print statements should be removed or disabled after debugging.

## A4 – Conclusion

A reliable debugging plan combines small reproducible test files, assertions, structured logging and a debugger. Performance should be measured at both pipeline level and function level using `perf_counter`, `timeit` and `cProfile`. Optimisation should target measured bottlenecks rather than guesswork.